In [ ]:
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import datasets, transforms

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
mnist_transform = transforms.ToTensor()
cifar_transform = transforms.ToTensor()

mnist_full_train = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)

cifar_full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)

print("MNIST train size:", len(mnist_full_train))
print("MNIST test size:", len(mnist_test))
print("CIFAR train size:", len(cifar_full_train))
print("CIFAR test size:", len(cifar_test))

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.07MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.19MB/s]
100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


MNIST train size: 60000
MNIST test size: 10000
CIFAR train size: 50000
CIFAR test size: 10000


In [ ]:
mnist_train, mnist_val = random_split(mnist_full_train, [50000, 10000])
cifar_train, cifar_val = random_split(cifar_full_train, [45000, 5000])

print("MNIST train:", len(mnist_train), "val:", len(mnist_val))
print("CIFAR train:", len(cifar_train), "val:", len(cifar_val))

MNIST train: 50000 val: 10000
CIFAR train: 45000 val: 5000


In [ ]:
def make_loaders(train_dataset, val_dataset, test_dataset, batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = make_loaders(mnist_train, mnist_val, mnist_test, batch_size=64)

images, labels = next(iter(train_loader))
print("MNIST batch image shape:", images.shape)
print("MNIST batch label shape:", labels.shape)

train_loader, val_loader, test_loader = make_loaders(cifar_train, cifar_val, cifar_test, batch_size=64)

images, labels = next(iter(train_loader))
print("CIFAR batch image shape:", images.shape)
print("CIFAR batch label shape:", labels.shape)

MNIST batch image shape: torch.Size([64, 1, 28, 28])
MNIST batch label shape: torch.Size([64])
CIFAR batch image shape: torch.Size([64, 3, 32, 32])
CIFAR batch label shape: torch.Size([64])


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, num_classes=10, dropout=0.2):
        super().__init__()

        layers = []
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, num_classes))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten image
        return self.network(x)

In [ ]:
mnist_mlp = MLP(input_size=784, hidden_layers=[128], dropout=0.2).to(device)
cifar_mlp = MLP(input_size=3072, hidden_layers=[512, 256, 128], dropout=0.5).to(device)

x_mnist = torch.randn(64, 1, 28, 28).to(device)
x_cifar = torch.randn(64, 3, 32, 32).to(device)

print("MNIST MLP output:", mnist_mlp(x_mnist).shape)
print("CIFAR MLP output:", cifar_mlp(x_cifar).shape)

MNIST MLP output: torch.Size([64, 10])
CIFAR MLP output: torch.Size([64, 10])


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 8, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        if in_channels == 1:   # MNIST: 28x28 -> 14x14 -> 7x7
            self.classifier = nn.Linear(8 * 7 * 7, num_classes)
        else:                  # CIFAR: 32x32 -> 16x16 -> 8x8
            self.classifier = nn.Linear(8 * 8 * 8, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
class EnhancedCNN(nn.Module):
    def __init__(self, in_channels, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        if in_channels == 1:   # MNIST: 28 -> 14 -> 7 -> 3
            self.classifier = nn.Sequential(
                nn.Linear(64 * 3 * 3, 128),
                nn.ReLU(),
                nn.Linear(128, num_classes)
            )
        else:                  # CIFAR: 32 -> 16 -> 8 -> 4
            self.classifier = nn.Sequential(
                nn.Linear(64 * 4 * 4, 128),
                nn.ReLU(),
                nn.Linear(128, num_classes)
            )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
mnist_simple = SimpleCNN(in_channels=1).to(device)
mnist_enhanced = EnhancedCNN(in_channels=1).to(device)

cifar_simple = SimpleCNN(in_channels=3).to(device)
cifar_enhanced = EnhancedCNN(in_channels=3).to(device)

x_mnist = torch.randn(64, 1, 28, 28).to(device)
x_cifar = torch.randn(64, 3, 32, 32).to(device)

print("MNIST SimpleCNN:", mnist_simple(x_mnist).shape)
print("MNIST EnhancedCNN:", mnist_enhanced(x_mnist).shape)
print("CIFAR SimpleCNN:", cifar_simple(x_cifar).shape)
print("CIFAR EnhancedCNN:", cifar_enhanced(x_cifar).shape)

MNIST SimpleCNN: torch.Size([64, 10])
MNIST EnhancedCNN: torch.Size([64, 10])
CIFAR SimpleCNN: torch.Size([64, 10])
CIFAR EnhancedCNN: torch.Size([64, 10])


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [ ]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, device, epochs=10, patience=3):
    best_val_acc = 0
    best_state = None
    history = []
    no_improve = 0

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(f"Epoch {epoch:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict()
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print("Early stopping triggered.")
            break

    runtime = time.time() - start_time

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_acc, runtime, history

In [ ]:
batch_size = 64
train_loader, val_loader, test_loader = make_loaders(mnist_train, mnist_val, mnist_test, batch_size)

model = MLP(input_size=784, hidden_layers=[128], dropout=0.2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model, best_val_acc, runtime, history = train_model(
    model, train_loader, val_loader, optimizer, criterion, device,
    epochs=5, patience=2
)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print("Best Val Acc:", best_val_acc)
print("Test Acc:", test_acc)
print("Runtime:", runtime)

Epoch 01 | Train Acc: 0.8888 | Val Acc: 0.9353
Epoch 02 | Train Acc: 0.9428 | Val Acc: 0.9529
Epoch 03 | Train Acc: 0.9584 | Val Acc: 0.9579
Epoch 04 | Train Acc: 0.9643 | Val Acc: 0.9630
Epoch 05 | Train Acc: 0.9705 | Val Acc: 0.9671
Best Val Acc: 0.9671
Test Acc: 0.9731
Runtime: 39.70143485069275


In [ ]:
import pandas as pd

In [ ]:
MLP_ARCHITECTURES = {
    "shallow": [128],
    "medium": [512, 256, 128],
    "deep": [1024, 512, 256, 128, 64]
}

In [ ]:
mlp_configs = [
    {"lr": 0.01,   "batch_size": 64,  "optimizer": "SGD",  "dropout": 0.2},
    {"lr": 0.01,   "batch_size": 64,  "optimizer": "Adam", "dropout": 0.2},
    {"lr": 0.001,  "batch_size": 64,  "optimizer": "SGD",  "dropout": 0.2},
    {"lr": 0.001,  "batch_size": 64,  "optimizer": "Adam", "dropout": 0.2},
    {"lr": 0.0001, "batch_size": 64,  "optimizer": "Adam", "dropout": 0.2},
    {"lr": 0.001,  "batch_size": 32,  "optimizer": "Adam", "dropout": 0.2},
    {"lr": 0.001,  "batch_size": 128, "optimizer": "Adam", "dropout": 0.2},
    {"lr": 0.001,  "batch_size": 64,  "optimizer": "Adam", "dropout": 0.5},
    {"lr": 0.01,   "batch_size": 128, "optimizer": "SGD",  "dropout": 0.5},
    {"lr": 0.0001, "batch_size": 32,  "optimizer": "Adam", "dropout": 0.5},
]

len(mlp_configs)

10

In [ ]:
def get_optimizer(name, model, lr, weight_decay=0.0):
    if name == "SGD":
        return optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif name == "Adam":
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

In [ ]:
def run_mlp_experiment(dataset_name, architecture_name, hidden_layers, config,
                       train_dataset, val_dataset, test_dataset,
                       epochs=10, patience=3):

    if dataset_name == "MNIST":
        input_size = 784
    elif dataset_name == "CIFAR10":
        input_size = 3072
    else:
        raise ValueError("dataset_name must be 'MNIST' or 'CIFAR10'")

    batch_size = config["batch_size"]
    train_loader, val_loader, test_loader = make_loaders(
        train_dataset, val_dataset, test_dataset, batch_size
    )

    model = MLP(
        input_size=input_size,
        hidden_layers=hidden_layers,
        dropout=config["dropout"]
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(config["optimizer"], model, lr=config["lr"])

    model, best_val_acc, runtime, history = train_model(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs=epochs, patience=patience
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    result = {
        "dataset": dataset_name,
        "architecture": architecture_name,
        "hidden_layers": str(hidden_layers),
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "optimizer": config["optimizer"],
        "dropout": config["dropout"],
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "runtime_sec": runtime
    }

    return result, history

In [ ]:
def tune_mlp_architecture(dataset_name, architecture_name, hidden_layers,
                          train_dataset, val_dataset, test_dataset,
                          configs, epochs=10, patience=3):

    all_results = []
    best_result = None
    best_val_acc = -1

    for i, config in enumerate(configs, start=1):
        print("=" * 80)
        print(f"{dataset_name} | MLP-{architecture_name} | Config {i}/{len(configs)}")
        print(config)

        result, history = run_mlp_experiment(
            dataset_name=dataset_name,
            architecture_name=architecture_name,
            hidden_layers=hidden_layers,
            config=config,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            epochs=epochs,
            patience=patience
        )

        all_results.append(result)

        print("Best Val Acc:", round(result["best_val_acc"], 4))
        print("Test Acc:", round(result["test_acc"], 4))
        print("Runtime (sec):", round(result["runtime_sec"], 2))

        if result["best_val_acc"] > best_val_acc:
            best_val_acc = result["best_val_acc"]
            best_result = result

    df = pd.DataFrame(all_results).sort_values(by="best_val_acc", ascending=False).reset_index(drop=True)
    return df, best_result

In [ ]:
mnist_shallow_df, mnist_shallow_best = tune_mlp_architecture(
    dataset_name="MNIST",
    architecture_name="shallow",
    hidden_layers=MLP_ARCHITECTURES["shallow"],
    train_dataset=mnist_train,
    val_dataset=mnist_val,
    test_dataset=mnist_test,
    configs=mlp_configs,
    epochs=8,
    patience=2
)

MNIST | MLP-shallow | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.6597 | Val Acc: 0.8429
Epoch 02 | Train Acc: 0.8398 | Val Acc: 0.8776
Epoch 03 | Train Acc: 0.8695 | Val Acc: 0.8908
Epoch 04 | Train Acc: 0.8825 | Val Acc: 0.8970
Epoch 05 | Train Acc: 0.8903 | Val Acc: 0.9025
Epoch 06 | Train Acc: 0.8969 | Val Acc: 0.9082
Epoch 07 | Train Acc: 0.9036 | Val Acc: 0.9115
Epoch 08 | Train Acc: 0.9089 | Val Acc: 0.9158
Best Val Acc: 0.9158
Test Acc: 0.9239
Runtime (sec): 60.87
MNIST | MLP-shallow | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.9132 | Val Acc: 0.9555
Epoch 02 | Train Acc: 0.9427 | Val Acc: 0.9588
Epoch 03 | Train Acc: 0.9501 | Val Acc: 0.9594
Epoch 04 | Train Acc: 0.9541 | Val Acc: 0.9611
Epoch 05 | Train Acc: 0.9560 | Val Acc: 0.9613
Epoch 06 | Train Acc: 0.9585 | Val Acc: 0.9629
Epoch 07 | Train Acc: 0.9590 | Val Acc: 0.9625
Epoch 08 | Train Acc: 0.9627 | Va

In [ ]:
mnist_shallow_df

,dataset,architecture,hidden_layers,lr,batch_size,optimizer,dropout,best_val_acc,test_acc,runtime_sec
0,MNIST,shallow,[128],0.0010,32,Adam,0.2,0.9733,0.9765,74.081670
1,MNIST,shallow,[128],0.0010,64,Adam,0.2,0.9718,0.9765,62.991311
2,MNIST,shallow,[128],0.0010,128,Adam,0.2,0.9698,0.9746,56.465903
3,MNIST,shallow,[128],0.0010,64,Adam,0.5,0.9689,0.9711,63.261419
4,MNIST,shallow,[128],0.0100,64,Adam,0.2,0.9641,0.9662,62.469495
5,MNIST,shallow,[128],0.0001,32,Adam,0.5,0.9405,0.9460,74.396330
6,MNIST,shallow,[128],0.0001,64,Adam,0.2,0.9358,0.9404,62.460160
7,MNIST,shallow,[128],0.0100,64,SGD,0.2,0.9158,0.9239,60.872779
8,MNIST,shallow,[128],0.0100,128,SGD,0.5,0.8982,0.9064,55.417305
9,MNIST,shallow,[128],0.0010,64,SGD,0.2,0.8250,0.8310,61.150095


In [ ]:
mnist_shallow_best

{'dataset': 'MNIST',
 'architecture': 'shallow',
 'hidden_layers': '[128]',
 'lr': 0.001,
 'batch_size': 32,
 'optimizer': 'Adam',
 'dropout': 0.2,
 'best_val_acc': 0.9733,
 'test_acc': 0.9765,
 'runtime_sec': 74.08166980743408}

In [ ]:
mnist_shallow_df.to_csv("mnist_mlp_shallow_tuning.csv", index=False)
print("Saved mnist_mlp_shallow_tuning.csv")

Saved mnist_mlp_shallow_tuning.csv


In [ ]:
mnist_medium_df, mnist_medium_best = tune_mlp_architecture(
    dataset_name="MNIST",
    architecture_name="medium",
    hidden_layers=MLP_ARCHITECTURES["medium"],
    train_dataset=mnist_train,
    val_dataset=mnist_val,
    test_dataset=mnist_test,
    configs=mlp_configs,
    epochs=8,
    patience=2
)

MNIST | MLP-medium | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.3354 | Val Acc: 0.5954
Epoch 02 | Train Acc: 0.6186 | Val Acc: 0.7960
Epoch 03 | Train Acc: 0.7836 | Val Acc: 0.8584
Epoch 04 | Train Acc: 0.8443 | Val Acc: 0.8846
Epoch 05 | Train Acc: 0.8690 | Val Acc: 0.8991
Epoch 06 | Train Acc: 0.8862 | Val Acc: 0.9084
Epoch 07 | Train Acc: 0.8987 | Val Acc: 0.9157
Epoch 08 | Train Acc: 0.9086 | Val Acc: 0.9219
Best Val Acc: 0.9219
Test Acc: 0.9266
Runtime (sec): 65.6
MNIST | MLP-medium | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.8767 | Val Acc: 0.9347
Epoch 02 | Train Acc: 0.9114 | Val Acc: 0.9331
Epoch 03 | Train Acc: 0.9177 | Val Acc: 0.9362
Epoch 04 | Train Acc: 0.9194 | Val Acc: 0.9432
Epoch 05 | Train Acc: 0.9192 | Val Acc: 0.9475
Epoch 06 | Train Acc: 0.9257 | Val Acc: 0.9524
Epoch 07 | Train Acc: 0.9287 | Val Acc: 0.9466
Epoch 08 | Train Acc: 0.9305 | Val A

In [ ]:
mnist_deep_df, mnist_deep_best = tune_mlp_architecture(
    dataset_name="MNIST",
    architecture_name="deep",
    hidden_layers=MLP_ARCHITECTURES["deep"],
    train_dataset=mnist_train,
    val_dataset=mnist_val,
    test_dataset=mnist_test,
    configs=mlp_configs,
    epochs=8,
    patience=2
)

MNIST | MLP-deep | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.1066 | Val Acc: 0.1829
Epoch 02 | Train Acc: 0.1265 | Val Acc: 0.1129
Epoch 03 | Train Acc: 0.1289 | Val Acc: 0.2094
Epoch 04 | Train Acc: 0.2691 | Val Acc: 0.3110
Epoch 05 | Train Acc: 0.4223 | Val Acc: 0.6306
Epoch 06 | Train Acc: 0.6461 | Val Acc: 0.7896
Epoch 07 | Train Acc: 0.7705 | Val Acc: 0.8407
Epoch 08 | Train Acc: 0.8378 | Val Acc: 0.8935
Best Val Acc: 0.8935
Test Acc: 0.894
Runtime (sec): 68.87
MNIST | MLP-deep | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.7980 | Val Acc: 0.8686
Epoch 02 | Train Acc: 0.8406 | Val Acc: 0.9122
Epoch 03 | Train Acc: 0.8494 | Val Acc: 0.9264
Epoch 04 | Train Acc: 0.7315 | Val Acc: 0.7135
Epoch 05 | Train Acc: 0.3647 | Val Acc: 0.3263
Early stopping triggered.
Best Val Acc: 0.9264
Test Acc: 0.3182
Runtime (sec): 44.55
MNIST | MLP-deep | Config 3/10
{'lr': 0.001, 'bat

In [ ]:
mnist_mlp_best_df = pd.DataFrame([
    mnist_shallow_best,
    mnist_medium_best,
    mnist_deep_best
])

mnist_mlp_best_df

,dataset,architecture,hidden_layers,lr,batch_size,optimizer,dropout,best_val_acc,test_acc,runtime_sec
0,MNIST,shallow,[128],0.001,32,Adam,0.2,0.9733,0.9765,74.081670
1,MNIST,medium,"[512, 256, 128]",0.001,64,Adam,0.2,0.9790,0.9784,66.042226
2,MNIST,deep,"[1024, 512, 256, 128, 64]",0.001,64,Adam,0.2,0.9772,0.9806,70.815244


In [ ]:
mnist_mlp_best_df.to_csv("mnist_mlp_best_results.csv", index=False)

In [ ]:
cifar_shallow_df, cifar_shallow_best = tune_mlp_architecture(
    dataset_name="CIFAR10",
    architecture_name="shallow",
    hidden_layers=MLP_ARCHITECTURES["shallow"],
    train_dataset=cifar_train,
    val_dataset=cifar_val,
    test_dataset=cifar_test,
    configs=mlp_configs,
    epochs=10,
    patience=2
)

CIFAR10 | MLP-shallow | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.2615 | Val Acc: 0.3104
Epoch 02 | Train Acc: 0.3320 | Val Acc: 0.3142
Epoch 03 | Train Acc: 0.3577 | Val Acc: 0.3404
Epoch 04 | Train Acc: 0.3749 | Val Acc: 0.3588
Epoch 05 | Train Acc: 0.3871 | Val Acc: 0.3846
Epoch 06 | Train Acc: 0.3997 | Val Acc: 0.3808
Epoch 07 | Train Acc: 0.4086 | Val Acc: 0.3620
Early stopping triggered.
Best Val Acc: 0.3846
Test Acc: 0.3585
Runtime (sec): 54.01
CIFAR10 | MLP-shallow | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.0993 | Val Acc: 0.0990
Epoch 02 | Train Acc: 0.0992 | Val Acc: 0.0994
Epoch 03 | Train Acc: 0.0995 | Val Acc: 0.0996
Epoch 04 | Train Acc: 0.0983 | Val Acc: 0.0986
Epoch 05 | Train Acc: 0.0984 | Val Acc: 0.0986
Early stopping triggered.
Best Val Acc: 0.0996
Test Acc: 0.1
Runtime (sec): 38.61
CIFAR10 | MLP-shallow | Config 3/10
{'lr': 0.001, 'batch_size'

In [ ]:
cifar_medium_df, cifar_medium_best = tune_mlp_architecture(
    dataset_name="CIFAR10",
    architecture_name="medium",
    hidden_layers=MLP_ARCHITECTURES["medium"],
    train_dataset=cifar_train,
    val_dataset=cifar_val,
    test_dataset=cifar_test,
    configs=mlp_configs,
    epochs=10,
    patience=2
)

CIFAR10 | MLP-medium | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.1444 | Val Acc: 0.2142
Epoch 02 | Train Acc: 0.2214 | Val Acc: 0.2522
Epoch 03 | Train Acc: 0.2742 | Val Acc: 0.2938
Epoch 04 | Train Acc: 0.3080 | Val Acc: 0.3074
Epoch 05 | Train Acc: 0.3291 | Val Acc: 0.3446
Epoch 06 | Train Acc: 0.3462 | Val Acc: 0.3588
Epoch 07 | Train Acc: 0.3634 | Val Acc: 0.3818
Epoch 08 | Train Acc: 0.3727 | Val Acc: 0.3682
Epoch 09 | Train Acc: 0.3819 | Val Acc: 0.3910
Epoch 10 | Train Acc: 0.3923 | Val Acc: 0.3132
Best Val Acc: 0.391
Test Acc: 0.3111
Runtime (sec): 79.24
CIFAR10 | MLP-medium | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.1767 | Val Acc: 0.2204
Epoch 02 | Train Acc: 0.1818 | Val Acc: 0.1764
Epoch 03 | Train Acc: 0.1667 | Val Acc: 0.1778
Early stopping triggered.
Best Val Acc: 0.2204
Test Acc: 0.1835
Runtime (sec): 24.42
CIFAR10 | MLP-medium | Config 3/10
{'lr':

In [ ]:
cifar_deep_df, cifar_deep_best = tune_mlp_architecture(
    dataset_name="CIFAR10",
    architecture_name="deep",
    hidden_layers=MLP_ARCHITECTURES["deep"],
    train_dataset=cifar_train,
    val_dataset=cifar_val,
    test_dataset=cifar_test,
    configs=mlp_configs,
    epochs=10,
    patience=2
)

CIFAR10 | MLP-deep | Config 1/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.1019 | Val Acc: 0.0968
Epoch 02 | Train Acc: 0.1176 | Val Acc: 0.1636
Epoch 03 | Train Acc: 0.1455 | Val Acc: 0.1560
Epoch 04 | Train Acc: 0.1719 | Val Acc: 0.1810
Epoch 05 | Train Acc: 0.1845 | Val Acc: 0.1600
Epoch 06 | Train Acc: 0.1902 | Val Acc: 0.1932
Epoch 07 | Train Acc: 0.1986 | Val Acc: 0.2032
Epoch 08 | Train Acc: 0.2061 | Val Acc: 0.1324
Epoch 09 | Train Acc: 0.2156 | Val Acc: 0.2146
Epoch 10 | Train Acc: 0.2251 | Val Acc: 0.1862
Best Val Acc: 0.2146
Test Acc: 0.193
Runtime (sec): 83.54
CIFAR10 | MLP-deep | Config 2/10
{'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout': 0.2}
Epoch 01 | Train Acc: 0.1417 | Val Acc: 0.1696
Epoch 02 | Train Acc: 0.1475 | Val Acc: 0.1750
Epoch 03 | Train Acc: 0.1544 | Val Acc: 0.1836
Epoch 04 | Train Acc: 0.1532 | Val Acc: 0.1556
Epoch 05 | Train Acc: 0.1394 | Val Acc: 0.1092
Early stopping triggered.
Best Val A

In [ ]:
cifar_mlp_best_df = pd.DataFrame([
    cifar_shallow_best,
    cifar_medium_best,
    cifar_deep_best
])

cifar_mlp_best_df

,dataset,architecture,hidden_layers,lr,batch_size,optimizer,dropout,best_val_acc,test_acc,runtime_sec
0,CIFAR10,shallow,[128],0.0010,128,Adam,0.2,0.4360,0.4363,70.780890
1,CIFAR10,medium,"[512, 256, 128]",0.0001,64,Adam,0.2,0.4696,0.4704,82.211029
2,CIFAR10,deep,"[1024, 512, 256, 128, 64]",0.0001,64,Adam,0.2,0.4504,0.4520,85.229198


In [ ]:
mnist_shallow_df.to_csv("mnist_mlp_shallow_tuning.csv", index=False)
mnist_medium_df.to_csv("mnist_mlp_medium_tuning.csv", index=False)
mnist_deep_df.to_csv("mnist_mlp_deep_tuning.csv", index=False)

cifar_shallow_df.to_csv("cifar_mlp_shallow_tuning.csv", index=False)
cifar_medium_df.to_csv("cifar_mlp_medium_tuning.csv", index=False)
cifar_deep_df.to_csv("cifar_mlp_deep_tuning.csv", index=False)

mnist_mlp_best_df.to_csv("mnist_mlp_best_results.csv", index=False)
cifar_mlp_best_df.to_csv("cifar_mlp_best_results.csv", index=False)

print("All MLP CSVs saved.")

All MLP CSVs saved.


In [37]:
cnn_configs = [
    {"lr": 0.01,   "batch_size": 32,  "weight_decay": 0.0},
    {"lr": 0.01,   "batch_size": 64,  "weight_decay": 1e-4},
    {"lr": 0.01,   "batch_size": 128, "weight_decay": 5e-3},

    {"lr": 0.001,  "batch_size": 32,  "weight_decay": 0.0},
    {"lr": 0.001,  "batch_size": 64,  "weight_decay": 1e-4},
    {"lr": 0.001,  "batch_size": 128, "weight_decay": 5e-3},

    {"lr": 0.0001, "batch_size": 32,  "weight_decay": 0.0},
    {"lr": 0.0001, "batch_size": 64,  "weight_decay": 1e-4},
    {"lr": 0.0001, "batch_size": 128, "weight_decay": 5e-3},
]

len(cnn_configs)

9

In [38]:
def run_cnn_experiment(dataset_name, architecture_name, config,
                       train_dataset, val_dataset, test_dataset,
                       epochs=10, patience=3):

    if dataset_name == "MNIST":
        in_channels = 1
    elif dataset_name == "CIFAR10":
        in_channels = 3
    else:
        raise ValueError("dataset_name must be 'MNIST' or 'CIFAR10'")

    batch_size = config["batch_size"]
    train_loader, val_loader, test_loader = make_loaders(
        train_dataset, val_dataset, test_dataset, batch_size
    )

    if architecture_name == "simple":
        model = SimpleCNN(in_channels=in_channels).to(device)
    elif architecture_name == "enhanced":
        model = EnhancedCNN(in_channels=in_channels).to(device)
    else:
        raise ValueError("architecture_name must be 'simple' or 'enhanced'")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    model, best_val_acc, runtime, history = train_model(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs=epochs, patience=patience
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    result = {
        "dataset": dataset_name,
        "architecture": architecture_name,
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "weight_decay": config["weight_decay"],
        "best_val_acc": best_val_acc,
        "test_acc": test_acc,
        "runtime_sec": runtime
    }

    return result, history

In [39]:
def tune_cnn_architecture(dataset_name, architecture_name,
                          train_dataset, val_dataset, test_dataset,
                          configs, epochs=10, patience=3):

    all_results = []
    best_result = None
    best_val_acc = -1

    for i, config in enumerate(configs, start=1):
        print("=" * 80)
        print(f"{dataset_name} | CNN-{architecture_name} | Config {i}/{len(configs)}")
        print(config)

        result, history = run_cnn_experiment(
            dataset_name=dataset_name,
            architecture_name=architecture_name,
            config=config,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            epochs=epochs,
            patience=patience
        )

        all_results.append(result)

        print("Best Val Acc:", round(result["best_val_acc"], 4))
        print("Test Acc:", round(result["test_acc"], 4))
        print("Runtime (sec):", round(result["runtime_sec"], 2))

        if result["best_val_acc"] > best_val_acc:
            best_val_acc = result["best_val_acc"]
            best_result = result

    df = pd.DataFrame(all_results).sort_values(by="best_val_acc", ascending=False).reset_index(drop=True)
    return df, best_result

In [40]:
mnist_simple_cnn_df, mnist_simple_cnn_best = tune_cnn_architecture(
    dataset_name="MNIST",
    architecture_name="simple",
    train_dataset=mnist_train,
    val_dataset=mnist_val,
    test_dataset=mnist_test,
    configs=cnn_configs,
    epochs=8,
    patience=2
)

MNIST | CNN-simple | Config 1/9
{'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0}
Epoch 01 | Train Acc: 0.9581 | Val Acc: 0.9768
Epoch 02 | Train Acc: 0.9789 | Val Acc: 0.9788
Epoch 03 | Train Acc: 0.9824 | Val Acc: 0.9809
Epoch 04 | Train Acc: 0.9845 | Val Acc: 0.9797
Epoch 05 | Train Acc: 0.9858 | Val Acc: 0.9834
Epoch 06 | Train Acc: 0.9863 | Val Acc: 0.9825
Epoch 07 | Train Acc: 0.9880 | Val Acc: 0.9825
Early stopping triggered.
Best Val Acc: 0.9834
Test Acc: 0.9855
Runtime (sec): 73.81
MNIST | CNN-simple | Config 2/9
{'lr': 0.01, 'batch_size': 64, 'weight_decay': 0.0001}
Epoch 01 | Train Acc: 0.9539 | Val Acc: 0.9749
Epoch 02 | Train Acc: 0.9780 | Val Acc: 0.9766
Epoch 03 | Train Acc: 0.9818 | Val Acc: 0.9812
Epoch 04 | Train Acc: 0.9830 | Val Acc: 0.9804
Epoch 05 | Train Acc: 0.9839 | Val Acc: 0.9832
Epoch 06 | Train Acc: 0.9856 | Val Acc: 0.9791
Epoch 07 | Train Acc: 0.9844 | Val Acc: 0.9792
Early stopping triggered.
Best Val Acc: 0.9832
Test Acc: 0.9802
Runtime (sec): 58.61
M

In [41]:
mnist_simple_cnn_df

,dataset,architecture,lr,batch_size,weight_decay,best_val_acc,test_acc,runtime_sec
0,MNIST,simple,0.0100,32,0.0000,0.9834,0.9855,73.811355
1,MNIST,simple,0.0100,64,0.0001,0.9832,0.9802,58.606896
2,MNIST,simple,0.0010,32,0.0000,0.9828,0.9827,83.524770
3,MNIST,simple,0.0010,128,0.0050,0.9820,0.9839,50.927203
4,MNIST,simple,0.0010,64,0.0001,0.9813,0.9824,42.128740
5,MNIST,simple,0.0100,128,0.0050,0.9793,0.9738,57.508236
6,MNIST,simple,0.0001,32,0.0000,0.9742,0.9779,83.196182
7,MNIST,simple,0.0001,64,0.0001,0.9730,0.9746,67.807506
8,MNIST,simple,0.0001,128,0.0050,0.9672,0.9695,58.416486


In [42]:
mnist_simple_cnn_best

{'dataset': 'MNIST',
 'architecture': 'simple',
 'lr': 0.01,
 'batch_size': 32,
 'weight_decay': 0.0,
 'best_val_acc': 0.9834,
 'test_acc': 0.9855,
 'runtime_sec': 73.81135535240173}

In [43]:
mnist_enhanced_cnn_df, mnist_enhanced_cnn_best = tune_cnn_architecture(
    dataset_name="MNIST",
    architecture_name="enhanced",
    train_dataset=mnist_train,
    val_dataset=mnist_val,
    test_dataset=mnist_test,
    configs=cnn_configs,
    epochs=8,
    patience=2
)

MNIST | CNN-enhanced | Config 1/9
{'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0}
Epoch 01 | Train Acc: 0.9575 | Val Acc: 0.9752
Epoch 02 | Train Acc: 0.9809 | Val Acc: 0.9785
Epoch 03 | Train Acc: 0.9844 | Val Acc: 0.9836
Epoch 04 | Train Acc: 0.9872 | Val Acc: 0.9867
Epoch 05 | Train Acc: 0.9876 | Val Acc: 0.9789
Epoch 06 | Train Acc: 0.9896 | Val Acc: 0.9824
Early stopping triggered.
Best Val Acc: 0.9867
Test Acc: 0.9848
Runtime (sec): 72.33
MNIST | CNN-enhanced | Config 2/9
{'lr': 0.01, 'batch_size': 64, 'weight_decay': 0.0001}
Epoch 01 | Train Acc: 0.9530 | Val Acc: 0.9772
Epoch 02 | Train Acc: 0.9818 | Val Acc: 0.9818
Epoch 03 | Train Acc: 0.9834 | Val Acc: 0.9806
Epoch 04 | Train Acc: 0.9847 | Val Acc: 0.9751
Early stopping triggered.
Best Val Acc: 0.9818
Test Acc: 0.9756
Runtime (sec): 36.65
MNIST | CNN-enhanced | Config 3/9
{'lr': 0.01, 'batch_size': 128, 'weight_decay': 0.005}
Epoch 01 | Train Acc: 0.9357 | Val Acc: 0.9186
Epoch 02 | Train Acc: 0.9717 | Val Acc: 0.9587
Ep

In [44]:
cifar_simple_cnn_df, cifar_simple_cnn_best = tune_cnn_architecture(
    dataset_name="CIFAR10",
    architecture_name="simple",
    train_dataset=cifar_train,
    val_dataset=cifar_val,
    test_dataset=cifar_test,
    configs=cnn_configs,
    epochs=10,
    patience=2
)

CIFAR10 | CNN-simple | Config 1/9
{'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0}
Epoch 01 | Train Acc: 0.4753 | Val Acc: 0.5284
Epoch 02 | Train Acc: 0.5561 | Val Acc: 0.5650
Epoch 03 | Train Acc: 0.5763 | Val Acc: 0.5226
Epoch 04 | Train Acc: 0.5883 | Val Acc: 0.5892
Epoch 05 | Train Acc: 0.5986 | Val Acc: 0.5586
Epoch 06 | Train Acc: 0.6018 | Val Acc: 0.5892
Early stopping triggered.
Best Val Acc: 0.5892
Test Acc: 0.601
Runtime (sec): 62.78
CIFAR10 | CNN-simple | Config 2/9
{'lr': 0.01, 'batch_size': 64, 'weight_decay': 0.0001}
Epoch 01 | Train Acc: 0.4630 | Val Acc: 0.4666
Epoch 02 | Train Acc: 0.5595 | Val Acc: 0.5174
Epoch 03 | Train Acc: 0.5899 | Val Acc: 0.5216
Epoch 04 | Train Acc: 0.6058 | Val Acc: 0.5736
Epoch 05 | Train Acc: 0.6125 | Val Acc: 0.5842
Epoch 06 | Train Acc: 0.6174 | Val Acc: 0.5852
Epoch 07 | Train Acc: 0.6220 | Val Acc: 0.5726
Epoch 08 | Train Acc: 0.6208 | Val Acc: 0.5746
Early stopping triggered.
Best Val Acc: 0.5852
Test Acc: 0.5767
Runtime (sec): 68.5

In [45]:
cifar_enhanced_cnn_df, cifar_enhanced_cnn_best = tune_cnn_architecture(
    dataset_name="CIFAR10",
    architecture_name="enhanced",
    train_dataset=cifar_train,
    val_dataset=cifar_val,
    test_dataset=cifar_test,
    configs=cnn_configs,
    epochs=10,
    patience=2
)

CIFAR10 | CNN-enhanced | Config 1/9
{'lr': 0.01, 'batch_size': 32, 'weight_decay': 0.0}
Epoch 01 | Train Acc: 0.4256 | Val Acc: 0.4956
Epoch 02 | Train Acc: 0.5780 | Val Acc: 0.5248
Epoch 03 | Train Acc: 0.6321 | Val Acc: 0.5748
Epoch 04 | Train Acc: 0.6663 | Val Acc: 0.6192
Epoch 05 | Train Acc: 0.6907 | Val Acc: 0.6814
Epoch 06 | Train Acc: 0.7090 | Val Acc: 0.6416
Epoch 07 | Train Acc: 0.7229 | Val Acc: 0.6616
Early stopping triggered.
Best Val Acc: 0.6814
Test Acc: 0.6589
Runtime (sec): 81.14
CIFAR10 | CNN-enhanced | Config 2/9
{'lr': 0.01, 'batch_size': 64, 'weight_decay': 0.0001}
Epoch 01 | Train Acc: 0.4680 | Val Acc: 0.4420
Epoch 02 | Train Acc: 0.6269 | Val Acc: 0.4938
Epoch 03 | Train Acc: 0.6761 | Val Acc: 0.6230
Epoch 04 | Train Acc: 0.7068 | Val Acc: 0.6708
Epoch 05 | Train Acc: 0.7178 | Val Acc: 0.6252
Epoch 06 | Train Acc: 0.7325 | Val Acc: 0.6264
Early stopping triggered.
Best Val Acc: 0.6708
Test Acc: 0.6171
Runtime (sec): 54.69
CIFAR10 | CNN-enhanced | Config 3/9
{'lr

In [46]:
mnist_simple_cnn_df.to_csv("mnist_simple_cnn_tuning.csv", index=False)
mnist_enhanced_cnn_df.to_csv("mnist_enhanced_cnn_tuning.csv", index=False)
cifar_simple_cnn_df.to_csv("cifar_simple_cnn_tuning.csv", index=False)
cifar_enhanced_cnn_df.to_csv("cifar_enhanced_cnn_tuning.csv", index=False)

mnist_cnn_best_df = pd.DataFrame([mnist_simple_cnn_best, mnist_enhanced_cnn_best])
cifar_cnn_best_df = pd.DataFrame([cifar_simple_cnn_best, cifar_enhanced_cnn_best])

mnist_cnn_best_df.to_csv("mnist_cnn_best_results.csv", index=False)
cifar_cnn_best_df.to_csv("cifar_cnn_best_results.csv", index=False)

In [47]:
mnist_trainval = ConcatDataset([mnist_train, mnist_val])
cifar_trainval = ConcatDataset([cifar_train, cifar_val])

print("MNIST train+val size:", len(mnist_trainval))
print("CIFAR train+val size:", len(cifar_trainval))

MNIST train+val size: 60000
CIFAR train+val size: 50000


In [48]:
def make_final_loaders(trainval_dataset, test_dataset, batch_size):
    trainval_loader = DataLoader(trainval_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return trainval_loader, test_loader

In [49]:
def train_full_model(model, train_loader, criterion, optimizer, device, epochs=10):
    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc
        })
        print(f"Epoch {epoch:02d} | Train Acc: {train_acc:.4f}")

    runtime = time.time() - start_time
    return model, runtime, history

In [50]:
def retrain_best_mlp(row, trainval_dataset, test_dataset, epochs=10):
    dataset_name = row["dataset"]
    architecture_name = row["architecture"]

    input_size = 784 if dataset_name == "MNIST" else 3072
    hidden_layers = MLP_ARCHITECTURES[architecture_name]
    batch_size = int(row["batch_size"])
    lr = float(row["lr"])
    optimizer_name = row["optimizer"]
    dropout = float(row["dropout"])

    train_loader, test_loader = make_final_loaders(trainval_dataset, test_dataset, batch_size)

    model = MLP(
        input_size=input_size,
        hidden_layers=hidden_layers,
        dropout=dropout
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(optimizer_name, model, lr=lr)

    model, runtime, history = train_full_model(
        model, train_loader, criterion, optimizer, device, epochs=epochs
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    result = {
        "dataset": dataset_name,
        "architecture": architecture_name,
        "hidden_layers": str(hidden_layers),
        "lr": lr,
        "batch_size": batch_size,
        "optimizer": optimizer_name,
        "dropout": dropout,
        "final_test_acc": test_acc,
        "runtime_sec": runtime
    }

    return result, history

In [51]:
final_mlp_results = []

all_mlp_best_df = pd.concat([mnist_mlp_best_df, cifar_mlp_best_df], ignore_index=True)

for _, row in all_mlp_best_df.iterrows():
    dataset_name = row["dataset"]

    if dataset_name == "MNIST":
        trainval_dataset = mnist_trainval
        test_dataset = mnist_test
        epochs = 10
    else:
        trainval_dataset = cifar_trainval
        test_dataset = cifar_test
        epochs = 15

    print("=" * 80)
    print(f"Final retrain MLP | {dataset_name} | {row['architecture']}")

    result, history = retrain_best_mlp(
        row=row,
        trainval_dataset=trainval_dataset,
        test_dataset=test_dataset,
        epochs=epochs
    )

    final_mlp_results.append(result)

final_mlp_retrained_df = pd.DataFrame(final_mlp_results)
final_mlp_retrained_df

Final retrain MLP | MNIST | shallow
Epoch 01 | Train Acc: 0.9085
Epoch 02 | Train Acc: 0.9526
Epoch 03 | Train Acc: 0.9654
Epoch 04 | Train Acc: 0.9700
Epoch 05 | Train Acc: 0.9742
Epoch 06 | Train Acc: 0.9774
Epoch 07 | Train Acc: 0.9800
Epoch 08 | Train Acc: 0.9816
Epoch 09 | Train Acc: 0.9833
Epoch 10 | Train Acc: 0.9847
Final retrain MLP | MNIST | medium
Epoch 01 | Train Acc: 0.9091
Epoch 02 | Train Acc: 0.9632
Epoch 03 | Train Acc: 0.9728
Epoch 04 | Train Acc: 0.9781
Epoch 05 | Train Acc: 0.9815
Epoch 06 | Train Acc: 0.9830
Epoch 07 | Train Acc: 0.9857
Epoch 08 | Train Acc: 0.9869
Epoch 09 | Train Acc: 0.9884
Epoch 10 | Train Acc: 0.9893
Final retrain MLP | MNIST | deep
Epoch 01 | Train Acc: 0.8959
Epoch 02 | Train Acc: 0.9634
Epoch 03 | Train Acc: 0.9714
Epoch 04 | Train Acc: 0.9776
Epoch 05 | Train Acc: 0.9812
Epoch 06 | Train Acc: 0.9831
Epoch 07 | Train Acc: 0.9841
Epoch 08 | Train Acc: 0.9866
Epoch 09 | Train Acc: 0.9871
Epoch 10 | Train Acc: 0.9880
Final retrain MLP | CIFAR1

,dataset,architecture,hidden_layers,lr,batch_size,optimizer,dropout,final_test_acc,runtime_sec
0,MNIST,shallow,[128],0.0010,32,Adam,0.2,0.9803,101.754460
1,MNIST,medium,"[512, 256, 128]",0.0010,64,Adam,0.2,0.9836,88.885520
2,MNIST,deep,"[1024, 512, 256, 128, 64]",0.0010,64,Adam,0.2,0.9821,94.928338
3,CIFAR10,shallow,[128],0.0010,128,Adam,0.2,0.4381,111.700300
4,CIFAR10,medium,"[512, 256, 128]",0.0001,64,Adam,0.2,0.5036,130.750005
5,CIFAR10,deep,"[1024, 512, 256, 128, 64]",0.0001,64,Adam,0.2,0.5019,138.523438


In [52]:
final_mlp_retrained_df.to_csv("final_mlp_retrained_results.csv", index=False)
print("Saved final_mlp_retrained_results.csv")

Saved final_mlp_retrained_results.csv


In [53]:
def retrain_best_cnn(row, trainval_dataset, test_dataset, epochs=10):
    dataset_name = row["dataset"]
    architecture_name = row["architecture"]

    in_channels = 1 if dataset_name == "MNIST" else 3
    batch_size = int(row["batch_size"])
    lr = float(row["lr"])
    weight_decay = float(row["weight_decay"])

    train_loader, test_loader = make_final_loaders(trainval_dataset, test_dataset, batch_size)

    if architecture_name == "simple":
        model = SimpleCNN(in_channels=in_channels).to(device)
    elif architecture_name == "enhanced":
        model = EnhancedCNN(in_channels=in_channels).to(device)
    else:
        raise ValueError("Unknown CNN architecture")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    model, runtime, history = train_full_model(
        model, train_loader, criterion, optimizer, device, epochs=epochs
    )

    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    result = {
        "dataset": dataset_name,
        "architecture": architecture_name,
        "lr": lr,
        "batch_size": batch_size,
        "weight_decay": weight_decay,
        "final_test_acc": test_acc,
        "runtime_sec": runtime
    }

    return result, history

In [54]:
final_cnn_results = []

all_cnn_best_df = pd.concat([mnist_cnn_best_df, cifar_cnn_best_df], ignore_index=True)

for _, row in all_cnn_best_df.iterrows():
    dataset_name = row["dataset"]

    if dataset_name == "MNIST":
        trainval_dataset = mnist_trainval
        test_dataset = mnist_test
        epochs = 10
    else:
        trainval_dataset = cifar_trainval
        test_dataset = cifar_test
        epochs = 15

    print("=" * 80)
    print(f"Final retrain CNN | {dataset_name} | {row['architecture']}")

    result, history = retrain_best_cnn(
        row=row,
        trainval_dataset=trainval_dataset,
        test_dataset=test_dataset,
        epochs=epochs
    )

    final_cnn_results.append(result)

final_cnn_retrained_df = pd.DataFrame(final_cnn_results)
final_cnn_retrained_df

Final retrain CNN | MNIST | simple
Epoch 01 | Train Acc: 0.9595
Epoch 02 | Train Acc: 0.9781
Epoch 03 | Train Acc: 0.9815
Epoch 04 | Train Acc: 0.9833
Epoch 05 | Train Acc: 0.9853
Epoch 06 | Train Acc: 0.9857
Epoch 07 | Train Acc: 0.9867
Epoch 08 | Train Acc: 0.9873
Epoch 09 | Train Acc: 0.9879
Epoch 10 | Train Acc: 0.9880
Final retrain CNN | MNIST | enhanced
Epoch 01 | Train Acc: 0.9643
Epoch 02 | Train Acc: 0.9861
Epoch 03 | Train Acc: 0.9896
Epoch 04 | Train Acc: 0.9917
Epoch 05 | Train Acc: 0.9933
Epoch 06 | Train Acc: 0.9939
Epoch 07 | Train Acc: 0.9948
Epoch 08 | Train Acc: 0.9952
Epoch 09 | Train Acc: 0.9957
Epoch 10 | Train Acc: 0.9953
Final retrain CNN | CIFAR10 | simple
Epoch 01 | Train Acc: 0.4426
Epoch 02 | Train Acc: 0.5438
Epoch 03 | Train Acc: 0.5716
Epoch 04 | Train Acc: 0.5861
Epoch 05 | Train Acc: 0.5982
Epoch 06 | Train Acc: 0.6073
Epoch 07 | Train Acc: 0.6127
Epoch 08 | Train Acc: 0.6195
Epoch 09 | Train Acc: 0.6214
Epoch 10 | Train Acc: 0.6243
Epoch 11 | Train Acc:

,dataset,architecture,lr,batch_size,weight_decay,final_test_acc,runtime_sec
0,MNIST,simple,0.010,32,0.0000,0.9832,115.074626
1,MNIST,enhanced,0.001,64,0.0001,0.9894,100.358321
2,CIFAR10,simple,0.001,128,0.0050,0.6158,119.689862
3,CIFAR10,enhanced,0.001,64,0.0001,0.7440,147.724979


In [55]:
final_cnn_retrained_df.to_csv("final_cnn_retrained_results.csv", index=False)
print("Saved final_cnn_retrained_results.csv")

Saved final_cnn_retrained_results.csv


In [56]:
all_final_results_df = pd.concat(
    [final_mlp_retrained_df, final_cnn_retrained_df],
    ignore_index=True,
    sort=False
)

all_final_results_df.to_csv("all_final_results.csv", index=False)
all_final_results_df

,dataset,architecture,hidden_layers,lr,batch_size,optimizer,dropout,final_test_acc,runtime_sec,weight_decay
0,MNIST,shallow,[128],0.0010,32,Adam,0.2,0.9803,101.754460,NaN
1,MNIST,medium,"[512, 256, 128]",0.0010,64,Adam,0.2,0.9836,88.885520,NaN
2,MNIST,deep,"[1024, 512, 256, 128, 64]",0.0010,64,Adam,0.2,0.9821,94.928338,NaN
3,CIFAR10,shallow,[128],0.0010,128,Adam,0.2,0.4381,111.700300,NaN
4,CIFAR10,medium,"[512, 256, 128]",0.0001,64,Adam,0.2,0.5036,130.750005,NaN
5,CIFAR10,deep,"[1024, 512, 256, 128, 64]",0.0001,64,Adam,0.2,0.5019,138.523438,NaN
6,MNIST,simple,NaN,0.0100,32,NaN,NaN,0.9832,115.074626,0.0000
7,MNIST,enhanced,NaN,0.0010,64,NaN,NaN,0.9894,100.358321,0.0001
8,CIFAR10,simple,NaN,0.0010,128,NaN,NaN,0.6158,119.689862,0.0050
9,CIFAR10,enhanced,NaN,0.0010,64,NaN,NaN,0.7440,147.724979,0.0001
